# AI-Powered Student Analytics

**Mode:** Live  
**Level:** Capstone  
**Estimated time:** 45 minutes

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## Problem and outcome

Answer one bounded program-lead question with real OpenAI reasoning grounded in deterministic cohort metrics. The database tool computes facts; the model explains which cohort needs attention. The run is intentionally limited to one question to control cost and flakiness.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(), Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    get_events, require_env, setup_case_study, show_callout,
    show_flow, show_json, show_roles, show_spore, show_timeline, stage,
)

setup_case_study('AI-Powered Student Analytics')


## Course prerequisites

This capstone assumes: `course-09-model-runtime`, `course-10-hitl`, `case-study-student-analytics`. The implementation focuses on system design instead of explaining routine Python syntax.


## Architecture and responsibilities

A deterministic tool owns SQL and returns a complete cohort snapshot. ModelRuntime exposes its JSON schema to OpenAI and executes the provider-generated call. The Agent then reasons over the returned facts and produces a concise recommendation.


In [ ]:
show_roles([
('SQLite tool', 'Compute trusted metrics', 'tool'),
('ModelRuntime', 'Expose schema and execute call', 'reef'),
('OpenAI', 'Reason over returned facts', 'provider'),
('Program lead', 'Receive bounded recommendation', 'human')
])


## Design decisions

- SQL and metric definitions remain deterministic application code.
- The model must call the tool before answering and receives no raw SQL capability.
- One certification question keeps provider cost and behavioral surface bounded.
- Assertions prove tool execution, grounded result structure, and the expected cohort conclusion.


## Implementation

The next cells are grouped by subsystem. Each group exposes the Praval objects that matter to the architecture.


In [ ]:
import json
import sqlite3
import tempfile

from praval import Agent, ToolSpec

values = require_env("OPENAI_API_KEY", "PRAVAL_OPENAI_MODEL")
workspace = tempfile.TemporaryDirectory(prefix="praval-ai-analytics-")
database = str(Path(workspace.name) / "students.sqlite3")
connection = sqlite3.connect(database)
connection.execute(
    "CREATE TABLE outcomes (student TEXT, cohort TEXT, score REAL, completed INTEGER)"
)
connection.executemany(
    "INSERT INTO outcomes VALUES (?, ?, ?, ?)",
    [
        ("Ada", "alpha", 91.0, 1), ("Ben", "alpha", 83.0, 1),
        ("Cy", "beta", 64.0, 0), ("Dee", "beta", 72.0, 1),
    ],
)
connection.commit()
tool_executions = []


def cohort_snapshot(scope: str) -> str:
    if scope != "all":
        raise ValueError("this capstone only supports scope='all'")
    rows = connection.execute(
        "SELECT cohort, ROUND(AVG(score), 1), "
        "ROUND(AVG(completed) * 100, 1) FROM outcomes GROUP BY cohort"
    ).fetchall()
    snapshot = {
        cohort: {"average_score": score, "completion_rate": completion}
        for cohort, score, completion in rows
    }
    tool_executions.append(snapshot)
    stage("analytics tool", "snapshot computed", ", ".join(snapshot))
    return json.dumps(snapshot, sort_keys=True)


### Grounded Agent

The Agent receives one strict tool and no path for arbitrary database access.


In [ ]:
agent = Agent(
    "case-ai-student-analyst", provider="openai",
    model=values["PRAVAL_OPENAI_MODEL"],
    config={"temperature": 0, "max_output_tokens": 160, "timeout": 60},
)
agent.add_tool_spec(
    ToolSpec(
        name="cohort_snapshot",
        description="Return average score and completion rate for every cohort.",
        parameters={
            "type": "object",
            "properties": {"scope": {"type": "string", "enum": ["all"]}},
            "required": ["scope"], "additionalProperties": False,
        },
        strict=True, requires_approval=False,
        metadata={"category": "analytics", "data_source": "bounded fixture"},
    ),
    cohort_snapshot,
)
assert "cohort_snapshot" in agent.tools
show_json(agent.tools["cohort_snapshot"], "Grounding tool contract")


## End-to-end run

The provider must call the deterministic tool, identify the weaker cohort, and explain the evidence in one bounded response.


In [ ]:
question = (
    "You are advising a program lead. You must call cohort_snapshot with "
    "scope='all' before answering. Which cohort needs attention? Cite both "
    "average score and completion rate, then give one concise next action."
)
response = agent.generate(question)
assert len(tool_executions) == 1
assert set(tool_executions[0]) == {"alpha", "beta"}
assert tool_executions[0]["beta"] == {
    "average_score": 68.0, "completion_rate": 50.0,
}
assert "beta" in response.content.lower()
assert len(response.content.strip()) > 30
stage("OpenAI", "grounded recommendation", response.content)
show_json(
    {
        "question": question, "deterministic_snapshot": tool_executions[0],
        "model_answer": response.content,
        "usage": response.usage.model_dump() if response.usage else None,
        "provider": response.provider, "model": response.model,
    },
    "One bounded grounded analysis",
)


## Inspect the system

Inspect the evidence left by the completed run.


In [ ]:
show_json(
    {
        "tool_calls_executed": len(tool_executions),
        "facts_owned_by_code": tool_executions[0],
        "interpretation_owned_by_model": response.content,
    },
    "Deterministic facts versus model reasoning",
)
show_timeline()


## Failure modes and tradeoffs

- The recommendation is model-generated and may vary in phrasing even when the metric facts are fixed.
- The fixture proves grounding mechanics, not the validity of these metrics for a real education program.
- Tool outputs still need privacy controls before real student data reaches a provider.
- A provider authentication, capability, or schema error fails immediately; it is not a reason to bypass grounding.


## Extensions

- Add a reviewed intervention tool for recording the recommended action, with HITL approval enabled.
- Compare the model recommendation with a deterministic threshold policy and document disagreements.
- Try additional questions manually, but keep release certification bounded to the single maintained prompt.


## Cleanup

Release project resources explicitly.


In [ ]:
agent.close()
connection.close()
workspace.cleanup()
stage("case study", "cleanup complete", "agent and database released")
